# Part 3. Interpretation Layer

## Part Scope

This section reorganizes derived metrics into structured interpretation outputs.

The role of this layer is not to perform another round of calculation.  
The role is to convert quantitative results into reviewable diagnostic objects, compact threshold-based summaries, and controlled narrative inputs for downstream explanation.

The code below highlights the interpretation-layer structure.  
Utility functions, repetitive language branches, and UI-only glue code are omitted for readability.

## Interpretation Layer Design

The interpretation layer sits between metric construction and final narrative generation.

At this stage, the workflow reorganizes the metric package into three output forms:

- structured diagnostic insights
- threshold-based summaries
- AI-ready narrative inputs

This separation prevents the narrative layer from operating directly on raw metrics alone.  
Instead, the workflow first creates explicit intermediate objects that can be reviewed, sorted, exported, and reused before any final explanation is generated.

In [ ]:
try:
    latest = _latest_snapshot(st.session_state.snapshots)
    st.session_state.valuation = compute_valuation(latest, st.session_state.price_info["close"])
    st.session_state.rules = simple_rules(st.session_state.valuation, st.session_state.growth, language=language)
except Exception as e:
    st.error(f"{T['valuation_error']}: {e}")
    return

try:
    insights = generate_insights(
        growth=st.session_state.growth,
        valuation=st.session_state.valuation,
        snapshots=st.session_state.snapshots,
        yoy_list=st.session_state.yoy_list,
        language=language,
    )
    st.session_state.insights = insights
    st.session_state.insight_markdown = insights_to_markdown(
        insights, language=language, include_evidence=True
    )
except Exception as e:
    st.error(f"{T['insight_error']}: {e}")
    return

## Structured Diagnostic Objects

The primary output of this layer is a list of structured diagnostic insight objects.

Each object contains a stable key, category, title, status, summary, evidence, risk note, and priority.  
This makes the interpretation layer more explicit than a single block of narrative text.

Because the insights are structured objects rather than free-form text, they can be sorted, filtered, exported, and reused across application, reporting, and storage layers.

In [ ]:
def _make_insight(
    *,
    key: str,
    category: str,
    title: str,
    status: str,
    summary: str,
    evidence: Dict[str, Any],
    risk_note: str = "",
    priority: int = 99,
) -> Dict[str, Any]:
    return {
        "key": key,
        "category": category,
        "title": title,
        "status": status,
        "summary": summary,
        "evidence": evidence,
        "risk_note": risk_note,
        "priority": priority,
    }


def generate_insights(
    growth: Optional[Dict[str, Any]],
    valuation: Optional[Dict[str, Any]],
    snapshots: Optional[List[Any]],
    yoy_list: Optional[List[Dict[str, Any]]] = None,
    language: str = "ko",
) -> List[Dict[str, Any]]:
    growth = growth or {}
    valuation = valuation or {}
    snapshots = snapshots or []
    yoy_latest = _latest_yoy(yoy_list or [])

    insights = [
        _insight_growth_coherence(growth, yoy_latest, language),
        _insight_per_share_transfer(growth, language),
        _insight_profitability_trend(growth, language),
        _insight_cash_conversion(growth, language),
        _insight_balance_sheet_resilience(growth, valuation, language),
        _insight_price_metric_context(growth, valuation, language),
        _insight_growth_quality_divergence(growth, language),
        _insight_dilution_risk(growth, snapshots, language),
        _insight_recent_momentum(yoy_latest, language),
    ]

    return sorted(
        insights,
        key=lambda x: (x.get("priority", 99), _status_rank(x.get("status", ""))),
    )

## Threshold-Based Summaries

Alongside structured insights, the workflow creates a compact threshold-based summary layer.

These summaries are not investment signals and are not intended to replace fuller analytical review.  
Their role is to convert selected metric ranges into short diagnostic messages that are easier to scan inside the application.

This layer acts as a lightweight bridge between raw numerical output and longer-form explanation.

In [ ]:
def simple_rules(valuation: Dict[str, Any], growth: Dict[str, Any], language: str = "en") -> list[str]:

    msgs: list[str] = []

    pe = valuation.get("P/E")
    pb = valuation.get("P/B")
    p_fcf = valuation.get("P/FCF")
    earnings_yield = valuation.get("EarningsYield")
    fcf_yield = valuation.get("FCFYield")

    rev_cagr = growth.get("rev_cagr")
    ni_cagr = growth.get("ni_cagr")
    fcf_cagr = growth.get("fcf_cagr")

    if language == "ko":
        if pe is not None and pe <= 15:
            msgs.append("현재 가격은 EPS 대비 상대적으로 낮은 배수 구간에 위치합니다. (P/E ≤ 15)")
        elif pe is not None and pe > 25:
            msgs.append("현재 가격은 EPS 대비 높은 배수 구간에 위치합니다. (P/E > 25)")

## Review-Oriented Insight Formatting

The structured insights are also converted into a review-friendly markdown representation.

At this stage, each insight preserves its title, status label, summary, evidence block, and risk note rather than collapsing into plain narrative text.  
This keeps the interpretation output easy to inspect in the application while preserving the original diagnostic structure.

## Narrative Input Design

The final role of the interpretation layer is to prepare a controlled input for narrative generation.

The prompt builder does not pass arbitrary raw values into the model.  
Instead, it assembles a structured analytical block that includes source-linked market data, long-term change metrics, profitability context, cash conversion signals, balance-sheet indicators, price-linked indicators, and threshold-based summaries.

This design constrains the narrative layer to explain a defined analytical package rather than inventing structure on its own.

In [ ]:
def build_integrated_analysis_prompt(
    ticker: str,
    price_info: dict,
    growth: dict,
    valuation: dict,
    rules: list[str],
    language: str = "en",
) -> str:
    base_block = f"""
[1] Source-linked market data
- Symbol: {price_info.get('symbol', '-')}
- Date  : {price_info.get('date', '-')}
- Close : {num(price_info.get('close'))}

[2] Long-term change metrics
- Revenue CAGR   : {pct(growth.get('rev_cagr'))}
- Net Income CAGR: {pct(growth.get('ni_cagr'))}
- FCF CAGR       : {pct(growth.get('fcf_cagr'))}

[6] Threshold-based summaries
{_rules_to_text(rules, language=language)}
"""

    if language == "ko":
        prompt = f"""
위 데이터를 바탕으로 한국어로 구조화된 설명을 작성하세요.

원칙:
- 숫자를 직접 인용해 근거를 제시할 것
- 데이터가 부족하면 부족하다고 명시할 것
- “고평가/저평가”, “매수/매도”, “추천” 같은 표현은 사용하지 말 것
- 예측이 아니라 현재 데이터 구조와 상태에 대한 해석으로 제한할 것
"""

## Downstream Reuse

The outputs of this layer are not consumed only by the narrative generator.

The same structured insights can be displayed in the application, exported into tabular form, and preserved in downstream storage.  
For that reason, the interpretation layer functions as a reusable intermediate output layer rather than as a terminal display step.

## Transition to the Next Layer

With the interpretation layer in place, the workflow can move from analytical structuring to delivery.

The next section covers how these outputs are organized in the application layer, exported for review, and preserved through queryable storage.